# Avcı Drone: Hibrit Hedef Takip (Colab)

YOLO + OSTrack + Kalman takip modülünü Colab GPU'sunda çalıştırır.

**Başlamadan önce:** Menü → **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** → Kaydet.

**Gereken 3 dosya** (bilgisayarında hazır olsun):
1. `hibrit_takip.zip`: proje kodu
2. YOLO modelin (`.pt`)
3. Videon (`.mp4`, `.avi`, `.mkv` ...)

OSTrack ağırlığını **sen indirmiyorsun**, 3. hücre resmi bağlantıdan otomatik indirir.

**Google Drive'da** `MyDrive/avci_drone/` klasörü aç ve içine şunları koy:
```
avci_drone/
├── hibrit_takip/ (klasör) veya hibrit_takip.zip
├── <senin modelin>.pt
└── <senin videon>.mp4
```
OSTrack ağırlığı ilk çalıştırmada bu klasöre otomatik indirilir. Sonraki oturumlarda tekrar indirilmez.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU yok! Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Dosyaları hazırla
Drive bağlanırken izin penceresi çıkar, hesabını seçip izin ver. Dosyalar `DRIVE_DIR` klasöründe otomatik bulunur.
Bir dosya bulunamazsa hücre "Dosya seç" düğmesiyle onu ister.

In [ ]:
import os, glob, shutil, zipfile
from google.colab import files

USE_DRIVE = True                                   # Drive yerine yükleme penceresi için False yap
DRIVE_DIR = '/content/drive/MyDrive/avci_drone'    # Drive'daki klasörün

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = DRIVE_DIR
else:
    DATA = '/content/veri'
os.makedirs(DATA, exist_ok=True)
OUT_DIR = f'{DATA}/sonuclar'
os.makedirs(OUT_DIR, exist_ok=True)

WORK         = '/content/hibrit_takip'
OSTRACK_ROOT = '/content/OSTrack'
OSTRACK_SAVE = '/content/ostrack_output'

def upload(aciklama):
    """Yükleme penceresi açar, yüklenen dosyayı DATA klasörüne taşır ve yolunu döndürür."""
    print(f'>>> {aciklama} seç:')
    up = files.upload()
    if not up:
        raise SystemExit('Dosya seçilmedi, hücreyi tekrar çalıştır.')
    name = next(iter(up))
    dst = f'{DATA}/{name}'
    if os.path.abspath(name) != os.path.abspath(dst):
        shutil.move(name, dst)
    print('Yüklendi:', dst)
    return dst

def find_one(patterns):
    for p in patterns:
        hits = sorted(glob.glob(f'{DATA}/**/{p}', recursive=True))
        hits = [h for h in hits if '/sonuclar/' not in h and '/hibrit_takip/' not in h]
        if hits:
            return hits[0]
    return None

# 1) Proje kodu: klasör veya zip
src = f'{DATA}/hibrit_takip'
if not os.path.isdir(src):
    z = find_one(['hibrit_takip*.zip']) or upload('hibrit_takip.zip')
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(src, WORK)

# 2) YOLO modeli ve 3) video
YOLO_WEIGHTS = find_one(['*.pt']) or upload('YOLO model dosyası (.pt)')
VIDEO = find_one(['*.mp4', '*.avi', '*.mkv', '*.mov', '*.ts']) or upload('Video dosyası')

# Klasörde birden fazla model/video varsa hangisini istediğini buraya elle yaz:
# YOLO_WEIGHTS = f'{DATA}/benim_modelim.pt'
# VIDEO = f'{DATA}/benim_videom.mp4'

print('\nKod   :', WORK, '(OK)' if os.path.isfile(f'{WORK}/hybrid_tracker.py') else '(YOK!)')
print('Model :', YOLO_WEIGHTS)
print('Video :', VIDEO)

## 3. Kurulum: OSTrack, bağımlılıklar ve OSTrack ağırlığı (otomatik indirilir)

In [ ]:
!pip install -q ultralytics easydict timm jpeg4py lmdb gdown

# OSTrack kodu + skor yaması + uyumluluk düzeltmeleri
![ -d {OSTRACK_ROOT} ] || git clone -q https://github.com/botaoye/OSTrack.git {OSTRACK_ROOT}
!python {WORK}/colab/prepare_ostrack.py --ostrack-root {OSTRACK_ROOT} --save-dir {OSTRACK_SAVE}

# OSTrack ağırlığı (vitb_256_mae_ce_32x4_ep300, ~350 MB): resmi OSTrack Google Drive klasöründen
ckpt_dst = f'{OSTRACK_SAVE}/checkpoints/train/ostrack/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar'
ckpt_drive = f'{DATA}/OSTrack_ep0300.pth.tar'
if not os.path.isfile(ckpt_dst):
    if os.path.isfile(ckpt_drive):
        shutil.copy(ckpt_drive, ckpt_dst)
    else:
        import subprocess
        subprocess.run(['gdown', '1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV', '-O', ckpt_dst])
        if USE_DRIVE and os.path.isfile(ckpt_dst):
            shutil.copy(ckpt_dst, ckpt_drive)   # bir dahaki oturumda tekrar indirmesin
assert os.path.isfile(ckpt_dst) and os.path.getsize(ckpt_dst) > 100e6, 'OSTrack ağırlığı inmedi! Aşağıdaki nota bak.'
print('OSTrack ağırlığı hazır:', round(os.path.getsize(ckpt_dst) / 1e6), 'MB')

> **İndirme hata verirse** (Google bazen "çok fazla indirme" kotası koyar): tarayıcıda
> https://drive.google.com/file/d/1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV/view adresini aç, sağ üstten **İndir** de.
> Sonra aşağıdaki hücreyi çalıştırıp indirdiğin `OSTrack_ep0300.pth.tar` dosyasını seç.

In [ ]:
# SADECE otomatik indirme başarısız olduysa çalıştır
p = upload('OSTrack_ep0300.pth.tar')
shutil.copy(p, ckpt_dst)
print('OK', round(os.path.getsize(ckpt_dst) / 1e6), 'MB')

## 4. Birim testler (OSTrack/YOLO gerektirmez)

In [ ]:
!cd {WORK} && python -m pytest tests -q

## 5. Model ve video kontrolü
Sınıf listesine ve eğitim `imgsz` değerine bak. 6. hücrede `--imgsz` ve gerekirse `--classes` olarak bunları kullanacaksın.

In [ ]:
from ultralytics import YOLO
import cv2, sys, numpy as np
m = YOLO(YOLO_WEIGHTS)
print('Sınıflar     :', m.names)
print('Eğitim imgsz :', m.overrides.get('imgsz'))
cap = cv2.VideoCapture(VIDEO)
print('Video        :', int(cap.get(3)), 'x', int(cap.get(4)), '|', round(cap.get(5), 1), 'fps |', int(cap.get(7)), 'kare')
cap.release(); del m

# OSTrack yükleme testi: skor 1.0 dışında bir sayıysa yama çalışıyor
sys.path.insert(0, WORK)
from hybrid_tracker import OSTrackWrapper
ost = OSTrackWrapper(OSTRACK_ROOT)
frame = np.random.randint(0, 255, (480, 640, 3), np.uint8)
ost.init(frame, (300, 200, 20, 12))
print('OSTrack      :', ost.update(frame))
del ost; torch.cuda.empty_cache()

## 6. Kısa deneme (ilk 300 kare)
`EK_AYAR` değişkenine kendi ayarlarını yaz:
- `--imgsz 1280`: modeli hangi boyutla eğittiysen
- `--classes 0`: modelde birden fazla sınıf varsa İHA sınıfının numarası
- Diğer ayarlar: `--det-conf 0.4 --verify-every 3 --reinit-iou 0.35` ... (liste için `!python {WORK}/run_on_video.py -h`)

In [ ]:
EK_AYAR = '--imgsz 640'      # <- kendi ayarların
!cd {WORK} && python run_on_video.py --video "{VIDEO}" --weights "{YOLO_WEIGHTS}" --ostrack-root {OSTRACK_ROOT} --device 0 --half --out-video /content/deneme.mp4 --csv /content/deneme.csv --max-frames 300 {EK_AYAR}

In [ ]:
# Sonuç videosunu notebook içinde izle
from IPython.display import Video
Video('/content/deneme_h264.mp4', embed=True, width=800)

## 7. Tüm video

In [ ]:
name = os.path.splitext(os.path.basename(VIDEO))[0]
!cd {WORK} && python run_on_video.py --video "{VIDEO}" --weights "{YOLO_WEIGHTS}" --ostrack-root {OSTRACK_ROOT} --device 0 --half --out-video "/content/{name}_sonuc.mp4" --csv "{OUT_DIR}/{name}_sonuc.csv" --log-level WARNING {EK_AYAR}
!cp "/content/{name}_sonuc_h264.mp4" "{OUT_DIR}/" && ls -lh "{OUT_DIR}"

In [ ]:
# Sonuçları bilgisayarına indir (Drive kullanıyorsan zaten Drive'da duruyor)
files.download(f'{OUT_DIR}/{name}_sonuc.csv')
files.download(f'{OUT_DIR}/{name}_sonuc_h264.mp4')

## 8. Sonuç analizi (parametre ayarı için)
Durumların zaman içindeki dağılımı ve OSTrack skoru. `track_score_min` eşiğini doğru takipteki skorlarla kopma anındaki skorların arasına koy.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{OUT_DIR}/{name}_sonuc.csv')
order = ['ARAMA', 'ADAY', 'TAKIP', 'KAYIP']
print(df['state'].value_counts().reindex(order, fill_value=0))

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(df['time_s'], df['state'].map({s: i for i, s in enumerate(order)}), drawstyle='steps-post')
ax[0].set_yticks(range(4)); ax[0].set_yticklabels(order); ax[0].set_title('Durum')
tk = df[df['state'] == 'TAKIP']
ax[1].scatter(tk['time_s'], tk['score'], s=2)
ax[1].axhline(0.35, color='r', ls='--', label='track_score_min')
ax[1].set_title('TAKIP skoru'); ax[1].set_xlabel('zaman (s)'); ax[1].legend()
plt.tight_layout(); plt.show()